# itero: a pomodoro

A pomodoro session as a workflow: an event says how long it lasts, a step waits that long, and a notification tells you it is over. Run the cells in order.

[Open in Colab](https://colab.research.google.com/github/aegea-dev/itero/blob/main/examples/pomodoro/pomodoro.ipynb): nothing to install on your machine.

## Install itero

The latest release, by the line the docs give: one static binary in `~/.local/bin`. A notebook has no desktop, so `#notify` writes each notification to a file instead.

In [ ]:
import os

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
# no desktop: notifications go to this file
os.environ["ITERO_NOTIFY_LOG"] = "/tmp/itero-notifications.log"
# no OS secret store here: secrets would come from .env
os.environ["ITERO_NO_SECRET_STORE"] = "1"
!command -v itero >/dev/null || curl -fsSL https://itero.sh/install | sh
!itero --version

## The example

Opened from a copy of the repository, the notebook works in its own folder; opened elsewhere (Colab), it takes the example from GitHub first.

In [ ]:
import pathlib, subprocess

if not pathlib.Path("workflows/pomodoro.json").exists():
    if not pathlib.Path("/tmp/itero").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/aegea-dev/itero", "/tmp/itero"], check=True)
    os.chdir("/tmp/itero/examples/pomodoro")
print(os.getcwd())
!cat workflows/pomodoro.json

Its one action, `actions/sleep`, waits as many seconds as its `duration` arg says; the built-in `#notify` then tells it is over.

In [ ]:
!cat actions/sleep

## Run it

Send the event, a session of 3 seconds, then let the daemon work until nothing is left to do: `--once` runs the workflow to its end and exits.

In [ ]:
!itero ws emit pomodoro.start '{"duration": 3}'
!itero daemon --once

## What happened

The notification, and the run: `done`.

In [ ]:
!cat "$ITERO_NOTIFY_LOG"
!itero ws wf runs

## Next

Change the duration and run again, or write a workflow of your own: [the docs](https://docs.itero.sh) start with one.